# Segment 4 · Robot learning with a surgical VLA and a world model
**CoRL 2026 Tutorial — Robot Learning on the Path to Surgical Autonomy** · minutes 45–60 · *v2*

In Segment 3 a small policy (ACT) learned SutureBot from scratch. Here we ask what large pretrained models add:

| | ACT (Segment 3) | **GR00T-H-N1.7** (VLA) | **Cosmos-H-Dreams** (world model) |
|---|---|---|---|
| Input | images + robot state | images + **language instruction** | one frame + a sequence of actions |
| Output | action chunk | action chunk (50 steps @ 30 Hz ≈ 1.7 s) | **video** of what those actions would do |
| Pretraining | none | Open-H: 20 robot platforms, 770 h | Open-H, 9 embodiments |
| Role today | baseline | the policy | a "simulator" to look at actions |

**How this notebook works**
- ▶ **Main thread** — the cells we run together. They call a shared GPU server, so this notebook runs on a free CPU Colab.
- 🧪 **Explore on your own** — the section at the bottom. Every cell there is independent; run any of them whenever you like.
- If the server is unreachable, set `BACKEND = "cache"` and every main-thread cell replays pre-computed results.

Models: [GR00T-H-N1.7](https://huggingface.co/nvidia/GR00T-H-N1.7) · [Cosmos-H-Dreams](https://huggingface.co/nvidia/Cosmos-H-Dreams) · Data: [Open-H-Embodiment / SutureBot](https://huggingface.co/datasets/nvidia/PhysicalAI-Robotics-Open-H-Embodiment)

## ▶ 0 · Setup (≈1 min)

In [ ]:
#@title Connect to the tutorial server { display-mode: "form" }
API_URL = "https://corl-seg4-5xptyrp9u.gobrev.dev"   #@param {type:"string"}
API_KEY = ""                          #@param {type:"string"}
BACKEND = "api"                       #@param ["api", "cache"]
CACHE_URL = "https://huggingface.co/datasets/kdb6df/corl2026-seg4-cache/resolve/main/seg4_bundle_{episode}.zip"  #@param {type:"string"}
EPISODE = 0                           #@param [0, 1441, 1442, 1443, 1444, 1445, 1446, 1447, 1448, 1449, 1450, 1451] {type:"raw"}
FRAME = -1                            #@param {type:"integer"}
EPISODE, FRAME = int(EPISODE), int(FRAME)   # Colab can pass form values as text
# EPISODE 0 = the demo clip (1441). 1441-1444 are needle throws, 1445-1451 knot tying; all are held out
# from both models and pre-computed. FRAME -1 = that clip's pre-computed start frame (returns instantly).

!pip -q install scikit-image imageio[ffmpeg] av huggingface_hub pyarrow jinja2 > /dev/null

import json, time, io, base64, os, numpy as np, pandas as pd, requests, matplotlib.pyplot as plt
from IPython.display import HTML, display
H = {"X-API-Key": API_KEY}

def _get(path):  return requests.get(API_URL + path, headers=H, timeout=60)
def _post(path, j, timeout=120):
    r = requests.post(API_URL + path, json=j, headers=H, timeout=timeout); r.raise_for_status(); return r.json()

if BACKEND == "api":
    try:
        h = requests.get(API_URL + "/health", timeout=10).json()
        print("✅ server up:", h["model"], "| episodes:", h["episodes"], "| cosmos queue:", h["cosmos_queue"])
        if EPISODE == 0: EPISODE = h["episodes"][0]
        if FRAME < 0: FRAME = int(h.get("start_frames", {}).get(str(EPISODE), 0))
    except Exception as e:
        print("⚠️ server unreachable -> switching to cached results.", e); BACKEND = "cache"

BUNDLE = None
if BACKEND == "cache":
    import urllib.request, zipfile
    if EPISODE == 0: EPISODE = 1441
    urllib.request.urlretrieve(CACHE_URL.format(episode=EPISODE), "bundle.zip")
    import shutil; shutil.rmtree("bundle", ignore_errors=True); zipfile.ZipFile("bundle.zip").extractall("bundle")
    BUNDLE = json.load(open("bundle/bundle.json")); EPISODE, FRAME = BUNDLE["episode"], BUNDLE["frame"]
print("backend:", BACKEND, "| episode:", EPISODE, "| start frame:", FRAME)

## ▶ 1 · The clip (≈1.5 min)
One **held-out test episode** from SutureBot (dVRK-Si, JHU). Neither model saw it in training. The default, 1441, is the same episode as Segment 3, so the ACT numbers are directly comparable. Want another? Pick any of 1441–1451 in the Setup cell and re-run from there: every one is pre-computed.

In [ ]:
#@title Download just this episode (parquet + endoscope video) { display-mode: "form" }
from huggingface_hub import hf_hub_download
REPO, ROOT = "nvidia/PhysicalAI-Robotics-Open-H-Embodiment", "Surgical/jhu/imerse/suturebot"
c = EPISODE // 1000
pq  = hf_hub_download(REPO, f"{ROOT}/data/chunk-{c:03d}/episode_{EPISODE:06d}.parquet", repo_type="dataset")
vid = hf_hub_download(REPO, f"{ROOT}/videos/chunk-{c:03d}/observation.images.endoscope.left/episode_{EPISODE:06d}.mp4", repo_type="dataset")
df = pd.read_parquet(pq)
GT = np.stack(df["action"].values).astype(np.float32)          # (T, 16) absolute @ 30 Hz
NAMES = [f"psm{a}_{k}" for a in (1, 2) for k in ("x","y","z","qx","qy","qz","qw","jaw")]
POS, JAWS = [0, 1, 2, 8, 9, 10], [7, 15]                    # x/y/z (m) and jaw (rad) columns
def unit(d):  return {"x": "mm", "y": "mm", "z": "mm", "jaw": "rad"}.get(NAMES[d].split("_")[1], "")
def scale(d): return 1000 if unit(d) == "mm" else 1            # plot positions in mm
TASKS = {0: "knot tying", 1: "needle pickup", 2: "needle throw"}
print(f"{len(GT)} frames ({len(GT)/30:.1f} s) | task: {TASKS[int(df.task_index.iloc[0])]}")
if "instruction.text" in df: print("instruction:", df["instruction.text"].iloc[0])

import av
def read_frames(path, idx=None, size=(512, 288)):
    out = []
    with av.open(path) as f:
        for i, fr in enumerate(f.decode(video=0)):
            if idx is None or i in idx: out.append(fr.to_ndarray(format="rgb24", width=size[0], height=size[1]))
            if idx is not None and i > max(idx): break
    return np.stack(out)

def show_video(path, width=640):
    if path is None: return
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width={width} controls loop autoplay muted src="data:video/mp4;base64,{b64}"></video>'))

def to_mp4(frames, path, fps=10):
    import imageio.v3 as iio; iio.imwrite(path, frames, fps=fps, codec="libx264"); return path

show_video(to_mp4(read_frames(vid, size=(480, 270))[::3], "clip.mp4"))

## ▶ 2 · Ask GR00T-H for an action chunk (≈3 min)
**In:** three camera views (endoscope + two wrist cameras) and a text instruction. (For this dVRK embodiment the model was trained with 100 % state dropout, so it does *not* use joint state.)
**Out:** 50 future steps × 16 values (xyz + quaternion + jaw for each arm), in the robot's own units.

In [ ]:
#@title GR00T-H inference { display-mode: "form" }
if BACKEND == "api":
    r = _post("/v1/groot/act", {"episode": EPISODE, "frame": FRAME})
else:
    r = BUNDLE["act"]
CHUNK = np.array(r["actions"])
print(f"instruction: {r['instruction']!r} | chunk {CHUNK.shape} | latency {r.get('latency_ms','cached')} ms | cached={r.get('cached')}")

t = np.arange(FRAME, FRAME + len(CHUNK)) / 30
fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
for j, (lab, dims) in enumerate([("PSM1 position", [0,1,2]), ("PSM2 position", [8,9,10]), ("jaws", [7,15])]):
    for d in dims:
        ax[j].plot(t, scale(d) * GT[FRAME:FRAME+len(CHUNK), d], "k-", lw=1, alpha=.6)
        ax[j].plot(t, scale(d) * CHUNK[:, d], "--", label=NAMES[d])
    ax[j].set_title(lab); ax[j].set_xlabel("time (s)"); ax[j].set_ylabel(unit(dims[0])); ax[j].legend(fontsize=7)
fig.suptitle("GR00T-H prediction (dashed) vs what the surgeon-operator actually did (black)"); plt.tight_layout()

## ▶ 3 · Open-loop evaluation, the same way NVIDIA does it (≈3 min)
We slide over the whole episode: every 50 steps, give GR00T-H the **real** observation, keep its 50-step chunk, and compare the stitched prediction with the recorded actions.
Metric = mean squared / absolute error over all 16 dimensions and all steps, exactly as in GR00T-H's official [`open_loop_eval.py`](https://github.com/NVIDIA-Medtech/GR00T-H/blob/main/gr00t/eval/open_loop_eval.py).
This is **open-loop**: errors never compound, because the model always sees the real scene. It says little about closed-loop success (see the Open-H report's discussion).
The **hold pose** row is a no-model reference (repeat the current pose for 50 steps). Surgical motion is slow, so it is a strong baseline over 1.7 s; beating it on the jaw, where the action is decisive, is harder than on position.


In [ ]:
#@title Open-loop MSE / MAE (GR00T-H vs ACT) { display-mode: "form" }
ACT_PRED_URL = ""  #@param {type:"string"}
r = _post("/v1/groot/open_loop", {"episode": EPISODE, "start": 0, "steps": 10_000}, timeout=600) if BACKEND == "api" else BUNDLE["open_loop"]
gt, pred = np.array(r["gt"]), np.array(r["pred"])
GROUPS = {"xyz (mm)": [0, 1, 2, 8, 9, 10], "quat": [3, 4, 5, 6, 11, 12, 13, 14], "jaw (rad)": [7, 15]}
def ol_metrics(gt, pred):  # MSE/MAE == open_loop_eval.py; per-group MAE added for readability
    pred = np.array(pred, copy=True)
    for s in (slice(3, 7), slice(11, 15)):  # q and -q are the same rotation: align signs first
        pred[np.sum(gt[:, s] * pred[:, s], 1) < 0, s] *= -1
    m = {"MSE": float(np.mean((gt - pred) ** 2)), "MAE": float(np.mean(np.abs(gt - pred)))}
    for g, ix in GROUPS.items():
        m[f"MAE {g}"] = float(np.mean(np.abs(gt[:, ix] - pred[:, ix]))) * (1000 if "mm" in g else 1)
    return m
hold = gt[np.arange(len(gt)) // 50 * 50]   # "do nothing": repeat the pose at each chunk start
rows = {"GR00T-H-N1.7": ol_metrics(gt, pred), "hold pose (no model)": ol_metrics(gt, hold)}
if ACT_PRED_URL:   # (T,16) .npy from Segment 3, same episode, same 50-step protocol
    act = np.load(io.BytesIO(requests.get(ACT_PRED_URL).content))[: len(gt)]
    rows["ACT (Segment 3)"] = ol_metrics(gt[: len(act)], act)
display(pd.DataFrame(rows).T.style.format("{:.4f}"))

fig, ax = plt.subplots(2, 4, figsize=(16, 5), sharex=True)
for i, d in enumerate([0, 1, 2, 7, 8, 9, 10, 15]):
    a = ax.flat[i]; a.plot(scale(d) * gt[:, d], "k", lw=1, label="GT"); a.plot(scale(d) * pred[:, d], "C1", lw=1, label="GR00T-H")
    for s in range(0, len(gt), 50): a.axvline(s, color="r", lw=.3)
    a.set_title(NAMES[d], fontsize=9); a.set_ylabel(unit(d), fontsize=8)
    if i >= 4: a.set_xlabel("step (30 Hz)", fontsize=8)
ax.flat[0].legend(); fig.suptitle("Open-loop rollout (red lines = new chunk predicted)"); plt.tight_layout()

## ▶ 4 · What would those actions *do*? Ask Cosmos-H-Dreams (≈3 min)
Cosmos-H-Dreams takes **one frame + a sequence of robot actions** and generates the video that should follow, 12 frames (~1.2 s) at a time.
We roll out 6 blocks (72 frames, ~7 s, the protocol of the Open-H report) twice from the same start frame:
1. with the **recorded actions** → any difference from the real video is *world-model error*;
2. with **GR00T-H's actions** → differences on top of that come from the *policy*.

Scoring = per-frame L1 and SSIM against the real video (Open-H report, Methods).

In [ ]:
#@title Generate (or fetch) the two rollouts { display-mode: "form" }
BLOCKS = 6
def dream(source, actions=None, frame=None, episode=None, instruction=None):
    if BACKEND != "api":
        print("🔌 This cell generates a new video, so it needs the live server (BACKEND = 'api')."); return None
    j = {"episode": episode or EPISODE, "frame": FRAME if frame is None else frame,
         "action_source": source, "blocks": BLOCKS, "instruction": instruction}
    if actions is not None: j["actions"] = np.asarray(actions).tolist()
    job = _post("/v1/cosmos/dream", j)
    t0 = time.time()
    while job.get("status") not in ("done", "error"):
        time.sleep(3); job = {**_get(f"/v1/jobs/{job['job_id']}").json(), "job_id": job["job_id"]}
        print(f"\r{source}: {job['status']} ({time.time()-t0:.0f}s)", end="")
    if job["status"] == "error": raise RuntimeError(job.get("error"))
    path = f"dream_{source}_{job['job_id']}.mp4"; open(path, "wb").write(_get(job["video"]).content)
    print(f"\r{source}: done{' (cached)' if job.get('cached') else ''}            "); return path

if BACKEND == "api":
    DREAM = {s: dream(s) for s in ("gt", "groot")}
else:
    DREAM = {s: "bundle/" + p for s, p in BUNDLE["dreams"].items()}

def load_dream(p):
    fr = read_frames(p)
    return fr[1:] if len(fr) == BLOCKS * 12 + 1 else fr   # drop context frame if included
real = read_frames(vid, idx=set(range(FRAME + 3, FRAME + 3 * BLOCKS * 12 + 1, 3)))
gen = {k: load_dream(p) for k, p in DREAM.items()}
n = min(len(real), *map(len, gen.values()))
panel = np.concatenate([real[:n], gen["gt"][:n], gen["groot"][:n]], axis=2)
print("left: real video | middle: Cosmos + recorded actions | right: Cosmos + GR00T-H actions")
show_video(to_mp4(panel, "side_by_side.mp4"), width=960)

In [ ]:
#@title Per-frame L1 / SSIM { display-mode: "form" }
from skimage.metrics import structural_similarity as ssim
def video_metrics(g, r):
    g, r = g.astype(np.float32) / 255, r.astype(np.float32) / 255
    l1 = np.abs(g - r).mean(axis=(1, 2, 3))
    s = np.array([ssim(a, b, channel_axis=-1, data_range=1.0) for a, b in zip(g, r)])
    return l1, s
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2)); summary = {}
for k, lab in [("gt", "recorded actions (world-model error)"), ("groot", "GR00T-H actions (+ policy error)")]:
    l1, s = video_metrics(gen[k][:n], real[:n]); summary[lab] = {"mean L1 ↓": l1.mean(), "mean SSIM ↑": s.mean()}
    ax[0].plot(l1, label=lab); ax[1].plot(s, label=lab)
for a, t in zip(ax, ["L1", "SSIM"]):
    a.set_title(t); a.set_xlabel("generated frame (~10 fps)"); [a.axvline(b * 12, color="gray", lw=.3) for b in range(BLOCKS)]
ax[0].legend(fontsize=8); plt.tight_layout(); display(pd.DataFrame(summary).T.style.format("{:.3f}"))

## ▶ 5 · Takeaways (≈1.5 min)
- **Pretraining helps** — in the Open-H report, GR00T-H matched ACT with 1/3 of the SutureBot data and was the only policy to complete end-to-end suturing (25 %).
- **Open-loop error is not task success.** Low MSE can still fail on the robot; small errors compound once the policy sees its own consequences.
- **World models let us *look* at a policy's actions without a robot**, but they have their own error. Always roll out recorded actions as a reference.
- **Next steps:** closed-loop evaluation inside the world model (the policy sees generated frames), simulation in [ORBIT-Surgical](https://orbit-surgical.github.io/), and fine-tuning GR00T-H on your own embodiment ([GR00T-H repo](https://github.com/NVIDIA-Medtech/GR00T-H)).
- **Run it yourself afterwards:** the same server is a one-click Brev Launchable (link on the tutorial website).

📄 Open-H-Embodiment consortium, *Open-H-Embodiment* report ([arXiv:2604.21017](https://arxiv.org/abs/2604.21017)).

---
# 🧪 Explore on your own
Each cell below stands alone (it only needs Setup and the clip above). Collapse this section during the live walkthrough.
Requests that are not pre-computed run live on the shared GPU and wait in a queue, so be patient with Cosmos jobs (~1 min each).

### 🧪 A · Change the instruction: does language steer the policy?
The first three choices are SutureBot's own task phrases, the only ones GR00T-H saw for this robot. The rest are off-script, and you can type your own. Expect small changes: the image tells the policy most of what to do, so this tests how much it listens to language at all.

In [ ]:
#@title Same frame, different instruction { display-mode: "form" }
INSTRUCTION = "knot tying"  #@param ["needle pickup", "needle throw", "knot tying", "do nothing", "move both arms out of view", "cut the suture", "open both jaws"] {allow-input: true}
if BACKEND == "cache":
    alt = BUNDLE["act_alt"].get(INSTRUCTION) or print("not in cache; needs the API")
else:
    alt = _post("/v1/groot/act", {"episode": EPISODE, "frame": FRAME, "instruction": INSTRUCTION})
if alt:
    A = np.array(alt["actions"])
    fig, ax = plt.subplots(1, 3, figsize=(15, 3))
    for j, (lab, dims) in enumerate([("PSM1 position", [0, 1, 2]), ("PSM2 position", [8, 9, 10]), ("jaws", JAWS)]):
        for k in dims:
            col = f"C{dims.index(k)}"   # one color per axis: solid = dataset instruction, dashed = yours
            ax[j].plot(scale(k) * CHUNK[:, k], "-", color=col, label=f"{NAMES[k]} (dataset)")
            ax[j].plot(scale(k) * A[:, k], "--", color=col, label=f"{NAMES[k]} ({INSTRUCTION})")
        ax[j].set_title(lab); ax[j].set_xlabel("step (30 Hz)"); ax[j].set_ylabel(unit(dims[0])); ax[j].legend(fontsize=6)
    dpos = 1000 * np.abs(A[:, POS] - CHUNK[:, POS]).mean(); djaw = np.abs(A[:, JAWS] - CHUNK[:, JAWS]).mean()
    plt.suptitle(f"dataset instruction vs {INSTRUCTION!r}: mean |Δ| position {dpos:.2f} mm, jaw {djaw:.3f} rad")
    plt.tight_layout()

### 🧪 B · Dream the instruction-steered actions

In [ ]:
#@title Cosmos rollout of a different instruction (live) { display-mode: "form" }
INSTRUCTION_B = "knot tying"  #@param ["needle pickup", "needle throw", "knot tying", "do nothing", "move both arms out of view", "cut the suture", "open both jaws"] {allow-input: true}
p = dream("groot", instruction=INSTRUCTION_B)
show_video(p)

### 🧪 C · Break the actions on purpose: is the world model paying attention?
Each choice edits the *recorded* actions, then Cosmos-H-Dreams renders them next to the real video and the unedited rollout. If the world model follows the actions, the scene should go visibly wrong. Positions are in metres in the data, so 0.01 = 1 cm. Some choices (e.g. *swap arms*, *5x the motion*) push Cosmos so far outside what it was trained on that the video smears or dissolves into noise: that is the world model failing, not a bug.

In [ ]:
#@title Perturb recorded actions and dream them (live) { display-mode: "form" }
PERTURBATION = "reverse the motion"  #@param ["freeze PSM1", "open both jaws", "snap both jaws open/shut", "swap arms", "double the motion", "5x the motion", "reverse the motion", "shift PSM1 by 1 cm in x", "shake both arms (±3 mm noise)", "twist PSM1 wrist 90°"]
n_raw = BLOCKS * 36 + 1
a = GT[FRAME:FRAME + n_raw].copy(); a0 = a[0].copy()
ramp = np.clip(np.arange(len(a)) / 15, 0, 1)[:, None]   # ease offsets in over 0.5 s instead of jumping

def qmul(p, q):  # Hamilton product, quaternions stored xyzw like the dataset
    x1, y1, z1, w1 = p.T; x2, y2, z2, w2 = q.T
    return np.stack([w1*x2 + x1*w2 + y1*z2 - z1*y2, w1*y2 - x1*z2 + y1*w2 + z1*x2,
                     w1*z2 + x1*y2 - y1*x2 + z1*w2, w1*w2 - x1*x2 - y1*y2 - z1*z2], 1)

if PERTURBATION == "freeze PSM1":            a[:, 0:8] = a0[0:8]
elif PERTURBATION == "open both jaws":       a[:, JAWS] = GT[:, JAWS].max(0)
elif PERTURBATION == "snap both jaws open/shut":   # full open <-> full shut every 0.5 s
    a[:, JAWS] = np.where((np.arange(len(a)) // 15 % 2)[:, None] == 0, GT[:, JAWS].max(0), GT[:, JAWS].min(0))
elif PERTURBATION == "swap arms":            a = np.concatenate([a[:, 8:], a[:, :8]], 1)
elif PERTURBATION == "double the motion":    a[:, POS] = a0[POS] + 2 * (a[:, POS] - a0[POS])
elif PERTURBATION == "5x the motion":        a[:, POS] = a0[POS] + 5 * (a[:, POS] - a0[POS])
elif PERTURBATION == "reverse the motion":   a[:, POS] = a0[POS] - (a[:, POS] - a0[POS])   # same path, opposite direction
elif PERTURBATION == "shift PSM1 by 1 cm in x": a[:, [0]] += 0.01 * ramp
elif PERTURBATION == "shake both arms (±3 mm noise)":
    a[:, POS] += np.random.default_rng(0).normal(0, 0.003, (len(a), len(POS)))
elif PERTURBATION == "twist PSM1 wrist 90°":  # rotate about the tool's own z-axis, eased in over 0.5 s
    th = np.deg2rad(90) * ramp[:, 0]
    a[:, 3:7] = qmul(a[:, 3:7], np.stack([0 * th, 0 * th, np.sin(th / 2), np.cos(th / 2)], 1))

ref = GT[FRAME:FRAME + n_raw]
move = max(np.linalg.norm(a[:, s] - ref[:, s], axis=1).max() for s in (slice(0, 3), slice(8, 11)))   # per arm, metres
rot = max(np.degrees(2 * np.arccos(np.clip(np.abs(np.sum(a[:, s] * ref[:, s], 1)), 0, 1))).max() for s in (slice(3, 7), slice(11, 15)))
print(f"{PERTURBATION}: largest change vs recorded: {1000 * move:.1f} mm position, {rot:.0f}° rotation, "
      f"{np.abs(a[:, JAWS] - ref[:, JAWS]).max():.2f} rad jaw")
p = dream("custom", actions=a)
if p:
    bad = load_dream(p); m = min(n, len(bad))
    print("left: real video | middle: Cosmos + recorded actions | right: Cosmos +", PERTURBATION)
    show_video(to_mp4(np.concatenate([real[:m], gen["gt"][:m], bad[:m]], axis=2), "perturbed.mp4"), width=960)

### 🧪 D · Another episode

In [ ]:
#@title Pick any episode hosted on the server { display-mode: "form" }
if BACKEND == "api":
    eps = _get("/v1/episodes").json(); print(eps["episodes"], eps.get("meta", ""))
OTHER_EPISODE = 0  #@param {type:"integer"}
OTHER_FRAME = 0    #@param {type:"integer"}
if OTHER_EPISODE:
    r = _post("/v1/groot/open_loop", {"episode": OTHER_EPISODE, "start": OTHER_FRAME, "steps": 300}, timeout=600)
    print("open-loop:", r["metrics"])

### 🧪 E · Bring your own policy (e.g. your ACT from Segment 3)

In [ ]:
#@title Dream any (T,16) absolute action array @30 Hz { display-mode: "form" }
MY_ACTIONS_NPY = ""  #@param {type:"string"}
# a local path or a URL; left empty, this dreams ACT's predictions from section 3 (ACT_PRED_URL)
src = MY_ACTIONS_NPY or ACT_PRED_URL
if src:
    mine = np.load(io.BytesIO(requests.get(src).content)) if src.startswith("http") else np.load(src)
    print("open-loop vs GT:", ol_metrics(GT[:len(mine)], mine))
    show_video(dream("custom", actions=mine[FRAME:FRAME + BLOCKS * 36 + 1]))

### 🧪 F · Homework: closed loop inside the world model
Feed Cosmos' generated frame back to the policy and repeat. Cosmos only renders the endoscope view, so use GR00T-H's endoscope-only embodiment (`jhu_imerse_dvrk_mono`) or a policy trained on the endoscope alone. This is how NVIDIA screened GR00T-H checkpoints (Cosmos-Surg-dVRK). Deploy the Launchable and try it.